# 호텔 룸 InstantSplat (Google Colab) — 사진 2~12장으로 3DGS

COLMAP 없이 소수의 사진만으로 가우시안 스플래팅을 만드는 [InstantSplat](https://instantsplat.github.io/) (NVIDIA) 파이프라인입니다.
2026-08-10 실전 검증 완료 (무료 T4, 사진 5장 성공. PyTorch 2.6+ 호환 패치 포함).

**사용 전 준비**
1. 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. 구글 드라이브 최상위에 `instantsplat_input` 폴더를 만들고 **사진 2~12장** 업로드
   - JPG/PNG만 인식 (아이폰 HEIC는 JPG로 변환)
   - 같은 방을 서로 다른 위치에서, 인접 사진끼리 시야가 절반 이상 겹치게
3. 셀을 위에서부터 순서대로 실행 (▶ 또는 Shift+Enter)

**소요 시간(무료 T4)**: 설치 ~15분 + 재구성 ~5분. 세션이 유휴로 초기화되면 셀 1부터 다시 실행해야 하므로 한 번에 쭉 진행하세요.

**주의**: 사진에 찍히지 않은 각도는 재현되지 않습니다. InstantSplat은 NVIDIA 연구용 라이선스이므로 상업 서비스에 쓰기 전 레포의 LICENSE를 확인하세요.

In [ ]:
# 1) GPU 확인 — "Tesla T4"가 보여야 합니다. 안 보이면 런타임 유형을 다시 확인하세요.
!nvidia-smi

In [ ]:
# 2) 레포 클론 + MASt3R 체크포인트 다운로드 (약 3~5분, 2.6GB)
#    rm -rf: 이전 세션 잔여물이 있어도 깨끗하게 다시 받기 위함
%cd /content
!rm -rf /content/InstantSplat
!git clone --recursive https://github.com/NVlabs/InstantSplat.git
%cd /content/InstantSplat
!mkdir -p mast3r/checkpoints/
!wget -q --show-progress https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth -P mast3r/checkpoints/

In [ ]:
# 3) 의존성 설치 + CUDA 확장 컴파일 (약 10분 — 경고(warning)는 무시)
#    마지막의 sed는 PyTorch 2.6+ 호환 패치: torch.load 기본값 변경으로 생기는
#    UnpicklingError를 방지 (신뢰된 공식 체크포인트이므로 안전)
%cd /content/InstantSplat
!pip install -q -r requirements.txt
!pip install -q ./submodules/simple-knn
!pip install -q ./submodules/diff-gaussian-rasterization
!pip install -q ./submodules/fused-ssim
!grep -rl "map_location='cpu')" /content/InstantSplat/mast3r /content/InstantSplat/dust3r --include=*.py | xargs -r sed -i "s/map_location='cpu')/map_location='cpu', weights_only=False)/g"
print("설치 + 패치 완료")

In [ ]:
# 4) 구글 드라이브 연결 + 사진 가져오기
SCENE = 'room1'  # ← 새 방/새 사진 세트마다 여기만 바꾸세요 (영문/숫자)

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob
INPUT_DIR = '/content/drive/MyDrive/instantsplat_input'
assert os.path.isdir(INPUT_DIR), 'instantsplat_input 폴더가 드라이브에 없습니다. 만들고 사진을 넣어주세요.'

SOURCE_PATH = f'/content/InstantSplat/assets/hotel/{SCENE}'
shutil.rmtree(SOURCE_PATH, ignore_errors=True)  # 이전 실행 잔여물 제거
os.makedirs(f'{SOURCE_PATH}/images')

imgs = sorted(glob.glob(f'{INPUT_DIR}/*.[jJpP][pPnN]*[gG]'))
assert 2 <= len(imgs) <= 12, f'사진이 {len(imgs)}장입니다. 2~12장(JPG/PNG)을 넣어주세요.'
for i, p in enumerate(imgs):
    shutil.copy(p, f'{SOURCE_PATH}/images/{i:03d}{os.path.splitext(p)[1].lower()}')

N_VIEWS = len(imgs)
MODEL_PATH = f'/content/InstantSplat/output_infer/hotel/{SCENE}/{N_VIEWS}_views'
GS_ITER = 1000  # 품질을 올리려면 2000~3000
print(f'[{SCENE}] 사진 {N_VIEWS}장 준비 완료')

In [ ]:
# 5) 1단계 — 기하 초기화 (MASt3R가 카메라 위치와 3D 포인트를 사진에서 직접 추정, 1~3분)
#    'cannot find cuda-compiled version of RoPE2D' 경고는 무시해도 됩니다
%cd /content/InstantSplat
!python ./init_geo.py -s {SOURCE_PATH} -m {MODEL_PATH} \
  --n_views {N_VIEWS} --focal_avg --co_vis_dsp --conf_aware_ranking --infer_video

In [ ]:
# 6) 2단계 — 가우시안 스플래팅 학습 (카메라 포즈 동시 최적화, 1~2분)
!python ./train.py -s {SOURCE_PATH} -m {MODEL_PATH} -r 1 \
  --n_views {N_VIEWS} --iterations {GS_ITER} --pp_optimizer --optim_pose

In [ ]:
# 7) 3단계 — 렌더링 (사진 사이를 부드럽게 이동하는 카메라 경로 영상 생성, 1~2분)
!python ./render.py -s {SOURCE_PATH} -m {MODEL_PATH} -r 1 \
  --n_views {N_VIEWS} --iterations {GS_ITER} --infer_video

In [ ]:
# 8) 결과를 드라이브에 저장 (장면 이름별로 저장되어 이전 결과를 덮어쓰지 않음)
import glob, shutil

plys = glob.glob(f'{MODEL_PATH}/**/point_cloud.ply', recursive=True)
assert plys, 'point_cloud.ply를 찾지 못했습니다. 5~7단계 출력의 오류를 확인하세요.'
shutil.copy(sorted(plys)[-1], f'/content/drive/MyDrive/instantsplat_{SCENE}.ply')
print(f'저장: MyDrive/instantsplat_{SCENE}.ply')

for i, v in enumerate(glob.glob(f'{MODEL_PATH}/**/*.mp4', recursive=True)):
    shutil.copy(v, f'/content/drive/MyDrive/instantsplat_{SCENE}_video_{i}.mp4')
    print(f'저장: MyDrive/instantsplat_{SCENE}_video_{i}.mp4')

## 9) 결과 보기

- `instantsplat_<장면>_video_0.mp4` — 드라이브에서 바로 재생해 보는 카메라 이동 영상
- `instantsplat_<장면>.ply` — **SuperSplat 편집기**(https://superspl.at/editor)에 드래그 → 마우스로 회전/확대하며 3D 확인.
  잡티(floater)는 박스 선택 후 Delete로 제거, `.splat` 압축 변환도 가능.

## 다른 사진으로 반복 작업

1. 드라이브 `instantsplat_input`의 기존 사진을 지우고 새 사진으로 교체
2. 셀 4의 `SCENE` 이름 변경 (예: `room2`) → 셀 4~8 재실행 (셀 1~3은 세션이 살아있으면 생략)

## 문제 해결

- **`init_geo.py: No such file or directory`** → 세션이 초기화된 것. 셀 2부터 다시 실행
- **`UnpicklingError: Weights only load failed`** → 셀 3의 패치(sed)가 실행되지 않은 것. 셀 3 재실행
- **`confidence_dsp.npy` 없음 (셀 6)** → 셀 5가 중간에 죽은 것. 셀 5 출력의 Traceback 확인
- **`CUDA out of memory`** → 사진 장수를 줄이거나(6~8장) 긴 변 1600px 이하로 리사이즈
- **결과가 찌그러짐/유령처럼 보임** → 사진 간 겹침 부족. 시점 차이가 큰 사진 제외 후 재실행